In [63]:
import cv2
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

In [64]:
# Build CNN (Convolutional Neural Network) Model
class AgePredictorCNN(nn.Module):
    def __init__(self):
        super(AgePredictorCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1) # Output 1 is age (Linear Regression Layer)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [65]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgePredictorCNN().to(device)

In [66]:
# Load Weights
model.load_state_dict(torch.load('age_prediction_model.pth', map_location=device))
model.eval()

AgePredictorCNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [67]:
# Transform
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [68]:
# Face Detection using Haar Cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

In [69]:
# OpenCV Webcam Capture
cap = cv2.VideoCapture(0)

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Change color space to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(40, 40))

    for (x, y, w, h) in faces:
        # Padding
        padding = 40
        x_pad = max(0, x - padding)
        y_pad = max(0, y - int(padding * 1.5))
        w_pad = min(frame.shape[1] - x_pad, w + (padding * 2))
        h_pad = min(frame.shape[0] - y_pad, h + int(padding * 2.5))

        # Square bounding box around the face
        cv2.rectangle(frame, (x_pad, y_pad), (x_pad + w_pad, y_pad + h_pad), (255, 0, 0), 2)
        
        # Face Only
        face_img = frame[y_pad:y_pad + h_pad, x_pad:x_pad + w_pad]
        
        try:
            # Convert BGR to RGB and prepare the image for the model
            face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(face_rgb)
            input_tensor = transform(pil_image).unsqueeze(0).to(device)
            
            # Predict
            with torch.no_grad():
                output = model(input_tensor)
                predicted_age = output.item()
            
            # Age
            text = f"Age: {predicted_age:.1f}"
            cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            
        except Exception as e:
            pass

    # Show the frame
    cv2.imshow('Age Prediction', frame)

    # Q to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press 'q' to quit.
